In [2]:
import uproot
import pandas as pd
#imports ^^^^
#uproot is the bridge between python and root
#pandas is a dataframework (note to self ask Dr. Zurek about this)

#Opens root file
df = uproot.open(
    "/volatile/clas12/cooperb/SULI/pid_training_test.root:PhysicsEvents"
).arrays(library="pd")

In [4]:
print("Shape:", df.shape) #prints rows and colums in that order

Shape: (1096573, 53)


In [5]:
print("\nEB-assigned PID counts:")
print(df["pid"].value_counts())       #prints a table showing what each track was reconstructed as


EB-assigned PID counts:
pid
 211    748271
-211    238328
 321     91897
-321     18077
Name: count, dtype: int64


In [6]:
print("\nMC-truth PID (top 15):") #this is just a header on this line
print(df["mc_matching_pid"].value_counts().head(15)) #shows how many were matched to generated particles
print(f"\nUnmatched tracks (no MC match): {(df['mc_matching_pid'] == -9999).sum()}") #shows how many failed to match


MC-truth PID (top 15):
mc_matching_pid
 211     665961
-211     216100
 321      46720
-9999     43028
 2        29237
 22       28316
 113      11365
 213       8382
 223       7810
-321       7379
 2212      3016
 323       2914
-213       2887
 111       2660
 11        2539
Name: count, dtype: int64

Unmatched tracks (no MC match): 43028


In [8]:
# this code prints the contamination matrix, the rows are the reconstructed particles the colums are the generated
#notice how a large amount of reconstructed kaons were actually pions.
top_truths = df["mc_matching_pid"].value_counts().head(8).index
contam = pd.crosstab(
    df["pid"],
    df["mc_matching_pid"].where(df["mc_matching_pid"].isin(top_truths), other="other"),
    margins=True,
)
print("\nContamination matrix (EB pid × MC-truth pid):")
print(contam)


Contamination matrix (EB pid × MC-truth pid):
mc_matching_pid  -9999    -211      2     22    113     211   213    321  \
pid                                                                        
-321              1027    7565    354    425    314      35    10     13   
-211              8254  207180   3039   3864   4245     698   259     34   
211              28725    1280  20360  19374   6157  639316  7302   2186   
321               5022      75   5484   4653    649   25912   811  44487   
All              43028  216100  29237  28316  11365  665961  8382  46720   

mc_matching_pid  other      All  
pid                              
-321              8334    18077  
-211             10755   238328  
211              23571   748271  
321               4804    91897  
All              47464  1096573  


In [9]:
print("\nKey feature stats:") #prints some useful information
print(df[["beta", "chi2pid", "rich_RQ"]].describe())
print(f"chi2pid sentinels (9999): {(df['chi2pid'] == 9999).sum()}")
print(f"Tracks with RICH hit (rich_RQ != -9999): {(df['rich_RQ'] != -9999).sum()}")


Key feature stats:
               beta       chi2pid       rich_RQ
count  1.096573e+06  1.096573e+06  1.096573e+06
mean   9.874587e-01  1.377903e+01 -9.174376e+03
std    2.229955e-02  3.693165e+02  2.750582e+03
min    3.494756e-01 -1.200003e+02 -9.999000e+03
25%    9.849973e-01 -6.809283e-01 -9.999000e+03
50%    9.924814e-01  5.534789e-02 -9.999000e+03
75%    9.972594e-01  7.902520e-01 -9.999000e+03
max    1.024288e+00  9.999000e+03  1.000000e+00
chi2pid sentinels (9999): 1498
Tracks with RICH hit (rich_RQ != -9999): 90432
